# Binary Classification of AOI Alarms into True Defects and False Calls: model training

Step-by-step walk-through of the binary classification of AOI alarms (true defect or false call) on the Siemens AOI data (Pfab et al. 2024).
Run `python scripts/download_data.py` once before this notebook.

1. Load the data
2. Train three models (scikit-learn): logistic regression (baseline), random forest, neural network
3. Set the classification threshold (at most 1 % of defective boards released)
4. Evaluate: confusion matrix, precision, recall
5. Day-by-day simulation of the line

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from aoi_classification.data import load, boards
from aoi_classification.model import MODELS, FalseCallClassifier
from aoi_classification.replay import replay, summary

## 1. Load the data
Each row is one solder joint flagged by the AOI machine. `class`: 0 = false call, 1 = true defect.

In [ ]:
df = load("data/dataset.csv")
b = boards(df)
print(f"{len(df):,} flagged joints on {len(b):,} boards; {b.defective.mean():.1%} of the boards are really defective")

## 2–3. Train the models and set the threshold
Train on the first 70 % of the days, test on the rest (time order, as on the real line).

In [ ]:
cut = df["day"].quantile(0.7)
train, test = df[df.day <= cut], df[df.day > cut]
clfs = {m: FalseCallClassifier(model=m).fit(train) for m in MODELS}

## 4. Evaluate
Positive class = defective board. Recall = share of defective boards sent to manual inspection (target ≥ 99 %).

In [ ]:
rows = []
for m, g in clfs.items():
    d = g.classify(test)
    pred = (~d.predicted_false_call).astype(int)   # 1 = predicted true defect
    tn, fp, fn, tp = confusion_matrix(d.defective, pred).ravel()
    rows.append({"model": m, "precision": precision_score(d.defective, pred), "recall": recall_score(d.defective, pred),
                 "false-call boards released": tn / (tn + fp), "defective boards released": fn / (fn + tp)})
pd.DataFrame(rows).round(3)

## 5. Day-by-day simulation
Each day is decided with a model trained only on earlier days. Full run: `python scripts/run_replay.py`.

In [ ]:
r = replay(df, start_day=int(df.day.quantile(0.5)) + 1, retrain_every=7, model="logistic_regression")
summary(r)